In [ ]:
# Metadata of different blocks
1. Read rdos_data csv file
2. Different types of filtering based on the dataframe created by block 1 e.g. country, attack duration
3. validate_origin(as_path) method to filter AS paths to only include those of the form "100 200 300" and with at least two elements.
4. Find BGP updates for a given prefix or less and in a given duration of a start and end time., and find its peers
5. Peers stored as an array
6. Function extract_as_numbers to read a text file containing DDoS mitigation tagged ASes: List from BGP.tools.
7. Find matching peers between no. 5 and no. 6
8. Find matching peers between no. 5 and a limited list of top scrubbers
9. Program to find prefixes that are announced (in BGP updates) and contains a scrubber ASN on their AS paths
10. Check manually the prefix announced time and the time seen by the RSDoS telescope
18. Find an immediate upstream provider checking conditions: Origin AS prepending and ASes belonging to the same organization
19. Find list of siblings of an ASN using ASRank API
20. Program to find siblings of an ASN based on CAIDA AS2Org data
21. Detects IP version v4 or v6 based on python ipaddress module

# 15 Oct: Currently working on block #10 and # 14

In [12]:
# Convert to datetime
df['t_start'] = pd.to_datetime(df['t_start'], errors='coerce')
df['t_end'] = pd.to_datetime(df['t_end'], errors='coerce')
df['hpdate'] = pd.to_datetime(df['hpdate'], errors='coerce').dt.date

# Drop rows where datetime conversion failed (optional)
df = df.dropna(subset=['t_start', 't_end', 'hpdate'])

# Filter: duration > 5 minutes AND hpdate == 2024-12-03
filtered_df = df[
    (df['t_end'] - df['t_start'] > pd.Timedelta(minutes=5)) &
    (df['hpdate'] == pd.to_datetime('2024-12-03').date())
]

# Show result
print(filtered_df)#-------------------BLOCK #1--------------------------------------------
# -------------------------------------------------------------------------

# This program contains the analysis of randomly spoofed (RS) DoS (https://www.caida.org/projects/stardust/docs/data/dos/)
# data received from CAIDA network telescope for the year 2020 to 2021-08-05
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/rdos_data_2020-2021.csv')
# df = df.loc[(df['target_ip'] == '178.18.93.169')]
df

,Unnamed: 0.1,Unnamed: 0,target_ip,nr_attacker_ips,nr_attacker_ports,nr_target_ports,nr_packets,nr_bytes,max_ppm,start_posix_time,end_posix_time,asn,country-code,continent-code,date
0,2751694,0,1.165.12.187,321,1,1,321,184896,50,1577920112,1577920620,3462,TW,AS,2020-01-01
1,2751695,1,1.165.189.159,146,146,3,122,30805,30,1577912409,1577915979,3462,TW,AS,2020-01-01
2,2751696,2,1.179.182.186,7,1,92,165,6840,38,1577921165,1577921519,131293,TH,AS,2020-01-01
3,2751697,3,1.30.218.10,23,6,61,63,2520,57,1577892521,1577892791,4837,CN,AS,2020-01-01
4,2751698,4,1.30.218.10,22,6,56,56,2240,54,1577921546,1577921774,4837,CN,AS,2020-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4981340,4257793,2783,96.44.123.162,4,1,451,502,20080,74,1628126444,1628128681,22995,CA,NaN,2021-08-05
4981341,4257794,2784,98.190.138.119,256,6,1083,989,39560,208,1628127557,1628128097,22773,US,NaN,2021-08-05
4981342,4257795,2785,98.232.0.138,11511,10599,1,11544,1000563,2075,1628126606,1628126960,7922,US,NaN,2021-08-05
4981343,4257796,2786,99.226.183.43,1556,1545,1,1561,132519,874,1628128034,1628128142,812,CA,NaN,2021-08-05


In [18]:
#-------------------BLOCK #2--------------------------------------------
# -------------------------------------------------------------------------

# Filter RSDoS data based on attack duration and country
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn' to suppress the warning message

# df.loc[df['country-code'] == "NL"]#.iloc[1:10]
# df2 = df.loc[(df['country-code'] == "NL")]
df2 = df
df2["attack_duration_minutes"] = (df2["end_posix_time"] - df2["start_posix_time"])/60

# Choose attack data of duration between 10 minutes and an hour (60 minutes)
condition = (df2['attack_duration_minutes'] >= 2) & (df2['attack_duration_minutes'] <= 60) 
df2 = df2.loc[condition]

# After sorting
df_sorted = df2.sort_values(['attack_duration_minutes', 'start_posix_time'], ascending=[False, False])
df_sorted.iloc[:10]

,Unnamed: 0.1,Unnamed: 0,target_ip,nr_attacker_ips,nr_attacker_ports,nr_target_ports,nr_packets,nr_bytes,max_ppm,start_posix_time,end_posix_time,asn,country-code,continent-code,date,attack_duration_minutes
4883751,4160204,1440,128.116.75.170,22280,5250,2,36780,1846584,1140,1627352765,1627356365,22697,US,NaN,2021-07-27,60.0
4875346,4151799,1088,117.27.239.149,24132,5504,5497,45027,2240750,1461,1627291093,1627294693,133774,CN,AS,2021-07-26,60.0
4872078,4148531,3748,210.237.55.170,59237,39146,512,71444,4000864,1295,1627175398,1627178998,7511,JP,AS,2021-07-25,60.0
4868109,4144562,1987,72.68.171.247,4472,1353,2,7884,395544,364,1627090689,1627094289,701,US,NaN,2021-07-24,60.0
4408242,3684695,3602,47.234.163.185,69389,1,1,71184,3416832,1377,1624269671,1624273271,11427,US,NaN,2021-06-21,60.0
4254301,3530754,9601,192.99.222.242,271416,64544,1,1413693,56547720,24237,1621859117,1621862717,16276,CA,NaN,2021-05-24,60.0
4254771,3531224,10071,209.58.185.123,183153,61720,1,260868,10867436,55024,1621840494,1621844094,133752,HK,AS,2021-05-24,60.0
4176925,3453378,12651,216.58.206.206,62,84,2,473,18920,53,1621495196,1621498796,15169,US,NaN,2021-05-20,60.0
4164232,3440685,18257,95.210.2.65,356,1,355,519,20760,38,1621384525,1621388125,29286,GB,EU,2021-05-19,60.0
4139776,3416229,5803,185.113.141.87,5596,5358,1,7691,429196,276,1621373202,1621376802,204094,PT,EU,2021-05-18,60.0


In [10]:
#-------------------BLOCK #3--------------------------------------------
# -------------------------------------------------------------------------

#  STEP 1: Helper function to filter AS paths to only include those of the form "100 200 300" 
# and with at least two elements.
# Returns bool: True if the AS path is valid, False otherwise.

import re

def validate_origin(as_path):
    valid = False
    # Check if the path matches the pattern: only space-separated numbers
    if re.fullmatch(r'(\d+\s)+\d+', as_path):
        # Split the path by spaces to count the elements
        elements = as_path.split()
        if len(elements) >= 2:
            valid = True
    return valid
    

In [16]:
#-------------------BLOCK #4--------------------------------------------
# -------------------------------------------------------------------------

# Check BGP updates (announcements) for a few targeted IP addresses in RSDoS file
import pybgpstream
import datetime

prefix = "128.116.75.170/32" 
     
# from_time = 1627345565
# until_time = 1627356365
 
from_time =  1627345565 # -7200 for 2 hours early
until_time = 1627363565 # + 7200 for 2 hours later

# Convert to UTC time
from_time_utc = datetime.datetime.utcfromtimestamp(from_time)
until_time_utc = datetime.datetime.utcfromtimestamp(until_time)

# Convert UTC datetime to string
from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

print("Time %s and %s" %(from_time_utc_str, until_time_utc_str))


stream = pybgpstream.BGPStream(
    from_time=from_time_utc_str, 
    until_time=until_time_utc_str,
    record_type="updates", # By default it is Update announcement 
    filter="prefix less "+prefix,
    project="ris",
    

  )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

p = [] # List containing immediate provider
o = [] # Origin ASN

print("Extracting records..")

   
# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
        pfx = elem.fields["prefix"]
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]

#         Convert as_path into list
        as_path_list = as_path.split()
        orig = as_path_list[-1]
        
        # List of 28 confirmed scrubbers from bgp.tools ddosm tags and DDoScovery paper.
        scrubbers = ['32787', '13335', '19551', '19905', '198949', '57724', '54113', '21859', '19324', 
                     '3223', '137409', '396998', '60068', '8220', '30456', '8757', '34309', '35280', '197068', 
                     '199524', '45474', '200020', '42649', '59796', '5405', '20052', ' 394009', '401073'] 
        
        single_data = []
        # Discard different forms of origins example AS-Set, confederation set/sequence. 
        # Take only a single AS origin which is common for a scrubbing activity
        if pfx != '0.0.0.0/0' and len(as_path_list) > 1:
            # Find second ASN in an AS path
            second_as = as_path_list[-2]
            # Extract its upstream provider to check if that one contains an scrubbers' ASN
            single_data.append({"prefix": pfx})
            single_data.append({"provider": second_as})
            single_data.append({"origin": orig})
            single_data.append({"time": time})
            single_data.append({"as_path": as_path_list})
            p.append(single_data)

# Save results into a csv file.
# Flatten each list of single-key dictionaries into a single dictionary
flattened_data = [{k: v for d in entry for k, v in d.items()} for entry in p]

# Write to CSV
with open('output.csv', 'w', newline='') as csvfile:
    fieldnames = ['prefix', 'provider', 'origin', 'time', 'as_path']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for row in flattened_data:
        writer.writerow(row)
        
providers = [entry[1]['provider'] for entry in p]
# Find number of peers that are listed as DDoS mitigation 
matching_elements = set(providers) & set(scrubbers)
print(matching_elements)
print("Total matching ASes ",len(matching_elements))

Time 2021-07-27 00:26:05 and 2021-07-27 05:26:05
Extracting records..
{'35280', '199524', '137409'}
Total matching ASes  3


In [ ]:
#-------------------BLOCK #5--------------------------------------------
# -------------------------------------------------------------------------
peers = ['39120', '20562', '200020', '6939', '24785', '63927']

In [5]:
#-------------------BLOCK #6--------------------------------------------
# -------------------------------------------------------------------------

# Function to read a text file containing DDoS mitigation tagged ASes: List from BGP.tools
def extract_as_numbers(file_path):
    as_numbers = []
    
    # Open the file in read mode
    with open(file_path, 'r') as file:
        # Read the file line by line
        for line in file:
            # Split each line by the comma
            as_info = line.split(",")
            # Extract the AS number and add it to the list
            as_number = as_info[0]  # Assuming AS number is the first item before the comma
            as_numbers.append(as_number)
    
    return as_numbers

# Example usage:
file_path = '/home/shyam/jupy/scrubber_activation/data/ddosm_22may.txt'  # Replace with your file path
ddosm_as_list = extract_as_numbers(file_path)

ddosm_as_list = [item.replace('AS', '') for item in ddosm_as_list]

print(len(ddosm_as_list))

28


In [23]:
#-------------------BLOCK #7--------------------------------------------
# -------------------------------------------------------------------------

# Find number of peers that are listed as DDoS mitigation 
matching_elements = set(peers) & set(ddosm_as_list)
print(matching_elements)
print("Total matching ASes ",len(matching_elements))

{'137409', '199524', '35280'}
Total matching ASes  3


In [ ]:
#-------------------BLOCK #8--------------------------------------------
# -------------------------------------------------------------------------

# Compare matching ASes with a list of top DDoS mitigation providers 
# {'AS40399', 'AS19551', 'AS10690', 'AS4528', 'AS12400', 'AS7018', 'AS20189', 'AS200020'}
top_ddos_providers = {'40399', '19551', '10690', '4528', '12400', '7018', '20189', '200020', '13335', '395747', 
                     '202623', '132892', '13335', '19905'}
matching_top_providers =  top_ddos_providers & set(peers)
print(matching_top_providers)

In [ ]:
#-------------------BLOCK #9--------------------------------------------
# -------------------------------------------------------------------------

# Program to understand the BGP announcements specific for potential scrubbers
# Note: All the helper methods (e.g. detect_ip_version) are at the bottom of the program file
# Step # 1
# Find customers (ASN and prefixes) of DDoS scrubber for a day
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import csv 

scrubber_asn = "200020"
stream = pybgpstream.BGPStream(
    from_time="2024-01-01 00:00:00",
    until_time="2020-01-31 23:59:59",
    collectors=["rrc00"],# "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",     
    filter = "path _"+scrubber_asn+"_" #Look for all the prefixes that contain AS200020 as an immediate provider
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'announcements')


prefix_details = [] # For storing ipv4 prefixes

# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Find upstream ASN in an AS path
        as_path = elem.fields["as-path"]
        as_path = as_path.split()
        orig = as_path[-1]
        
        # Call a function to find provider because provider might not always be the second last AS in case of
        # AS path prepending and sibling ASes
        second_as = find_immediate_provider(as_path)
        
        # Store origin asn and announcement time in a dictionary
        asn_time = {}
        
        # Condition that it is a scrub AS 
        # 1. AS should be the second last AS in an AS path
        # See all the prefixes that has AS200020 as a second last hop and only IPv4
        if second_as == scrubber_asn and detect_ip_version(pfx) == 'IPv4':
#             print("Prefix %s, asn %s , time %s elem %s" %(pfx, orig, time, elem))
#             print("### Prefix is %s" %pfx)
            asn_time["asn"] = orig
            asn_time["time"] = time
            asn_time["peer_asn"] = peer_asn
            asn_time["peer_ip"] = peer_ip
            asn_time["prefix"] = pfx
            prefix_details.append(asn_time)                   
print("Completed")

# Store the results into a csv file
csv_file = "/home/shyam/jupy/ddos_scrubber/data/as"+scrubber_asn+"_01_31_Dec_2020_rrc00.csv"

# Write to CSV
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=prefix_details[0].keys())
    writer.writeheader()
    writer.writerows(prefix_details)

print(f"Data has been written to {csv_file}")

In [ ]:
#-------------------BLOCK #10--------------------------------------------
# -------------------------------------------------------------------------

# Check manually the prefix announced time and the time seen by the RSDoS telescope
# Find the closest matching time for the prefix announced

import pandas as pd
csv_file = "/home/shyam/jupy/ddos_scrubber/data/as200020_01_31_Dec_2020_rrc00.csv"

df = pd.read_csv(csv_file)

# Check all the prefixes before an hour
condition = (df["prefix"] == "145.131.16.0/24") & (df["time"] >= 1608065123) 

df = df.loc[condition]
df_sorted = df.sort_values(['time'], ascending=[True])
df = df_sorted

df[["prefix","time"]].iloc[0]

In [ ]:
#-------------------BLOCK #11--------------------------------------------
# -------------------------------------------------------------------------

# Program to compare this prefix with list of RSDoS target IP addresses and then store attack start/end times
import pandas as pd
import ipaddress

# Read prefixes from a CSV file of (l1)
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/as200020_01_31_Dec_2020_rrc00.csv")
l1 = df["prefix"]


# Get unique prefixes
l1 = df['prefix'].unique()
print("Len %s" %len(l1))

# Convert prefixes to IPv4Network objects
prefixes = [ipaddress.IPv4Network(prefix) for prefix in l1]


# List of DDoS target IP addresses
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/rdos_data_2020-2021.csv')
# df = df.loc[df['country-code'] == "NL"]

# Select values from Dec 1 to Dec 31 GMT
condition = (df['start_posix_time'] >= 1606780800) & (df['end_posix_time'] <= 1609459199) 
df = df.loc[condition]

print("Total RDoS attack reported in that period is" ,len(df))


# Function to find the corresponding prefix for each IP address
def find_prefix(ip):
    ip_obj = ipaddress.IPv4Address(ip)
    flag = "0"
    for prefix in prefixes:
        if ip_obj in prefix:
            flag = str(prefix)
    return flag

# Apply the function to the DataFrame
df['Prefix'] = df['target_ip'].apply(find_prefix)
condition = df['Prefix'] != "0"

# Convert Unix timestamp to GMT
df['start_time_GMT'] = pd.to_datetime(df['start_posix_time'], unit='s', utc=True)
df['end_time_GMT'] = pd.to_datetime(df['end_posix_time'], unit='s', utc=True)
# Select and print four columns with the condition
result = df.loc[condition, ['target_ip', 'start_posix_time', 'start_time_GMT', 'end_time_GMT', 'end_posix_time','Prefix']]
print("Result is %s"%result)


In [ ]:
#-------------------BLOCK #12--------------------------------------------
# -------------------------------------------------------------------------

result.to_csv("/home/shyam/jupy/ddos_scrubber/data/matching_prefix_01_30_dec_2020_as200020.csv")
len(result)

In [ ]:
#-------------------BLOCK #13--------------------------------------------
# -------------------------------------------------------------------------

# Count the number of times a prefix is found within 01 to 15 July for AS200020
# Results stored in 
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/as200020_01_15_July_2021_rrc00.csv')

# Count the number of times each prefix is repeated
prefix_counts = df['prefix'].value_counts()
print("Prefix and their appearance counts are \n%s" %prefix_counts[-20: -10])

# Filter and print prefixes with counts equal to 1
# prefixes_with_repeatation_filter = prefix_counts[prefix_counts < 50]
# print(prefixes_with_repeatation_filter)

# Find unique prefixes
# unique_prefixes = df['prefix'].unique()

# Print unique prefixes
# print("No. of unique prefixes is %s" %len(unique_prefixes) +" and these are %s" %unique_prefixes)



In [ ]:
#-------------------BLOCK #14--------------------------------------------
# -------------------------------------------------------------------------

# Check the announcement pattern of prefixes that were seen by NBIP (AS200020),...
# ..in the file as200020_01_15_July_2021_rrc00.csv before and after certain hours
# Check BGP updates (announcements) for a few targeted IP addresses in RSDoS file
# Results are in office -> excel -> DDoS -> sheet -> NBIP 
import pybgpstream
import datetime
import pytricia


prefix = "87.250.130.0/24"
prefix_seen_time = 1608064467

# Please choose one of the two options at a time.
# 1. Check providers list before two hours the prefix is seen by a route collector 
# from_time = prefix_seen_time - 7201  
# until_time = prefix_seen_time - 1 

# # 2. Check if a victim IP adddress exists for a prefix one day before within -1 hr and + 2 hr of BGP update time
from_time = prefix_seen_time - 86400 - 7200 
until_time = prefix_seen_time - 86400# Check manually the prefix announced time and the time seen by the RSDoS telescope

import pandas as pd
csv_file = "/home/shyam/jupy/ddos_scrubber/data/as200020_01_31_Dec_2020_rrc00.csv"

df = pd.read_csv(csv_file)

# len(df)


# Convert to UTC time
from_time_utc = datetime.datetime.utcfromtimestamp(from_time)
until_time_utc = datetime.datetime.utcfromtimestamp(until_time)

# Convert UTC datetime to string
from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

print("Time %s and %s" %(from_time_utc_str, until_time_utc_str))


stream = pybgpstream.BGPStream(
    from_time=from_time_utc_str, until_time=until_time_utc_str,
    record_type="updates", # By default it is Update announcement 
    filter="prefix less "+prefix,
    project="ris"

  )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter("elemtype", "announcements")

pyt_v4 = pytricia.PyTricia() # For storing ipv4 prefixes

p = [] # List containing immediate provider
o = [] # Origin ASN

print("Extracting records..")

# Create a list of hash of prefix and provider
#[{"80.255.245.0/24": [200020]}, {"80.255.0.0/16": [49685, 174]}]

# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
        pfx = elem.fields["prefix"]
        
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]

        # Convert as_path into list
        as_path_list = as_path.split()
        orig = as_path_list[-1]
        
        # Store prefix and upstreams in a dictionary
        pfx_details = {}
        
        # Discard different forms of origins example AS-Set, confederation set/sequence. 
        # Take only a single AS origin which is common for a scrubbing activity
        if pfx != '0.0.0.0/0' and len(as_path_list) > 1:
           
        
#             Call a function to find provider because provider might not always be the second last AS in case of
#             AS path prepending and sibling ASes
              second_as = find_immediate_provider(as_path_list)
    
            # Find second ASN in an AS path
#             second_as = as_path_list[-2]
            
            # Extract its upstream provider to check if that one contains an scrubbers' ASN
            p.append(second_as)
            o.append(orig)
            
             # If the prefix is already in the tree, append the new provider
            if pyt_v4.has_key(pfx):
                pyt_v4[pfx]['provider'].append(second_as)
                
                # TODO: When I try to store origin here, it gives error
            else:
                # Add the prefix with a list of providers
                pyt_v4[pfx] = {'provider': [second_as]}
  
# Remove duplicates from provider lists (optional, if needed)
for prefix in pyt_v4:
    pyt_v4[prefix]['provider'] = list(set(pyt_v4[prefix]['provider']))

# Print prefixes and their providers
for prefix in pyt_v4:
    print (prefix,pyt_v4[prefix])
    

# Filter RSDoS data based on attack duration and check if a given prefix contains any IP address in that list
print("Finding matching IP address..")    

import ipaddress

pd.options.mode.chained_assignment = None  # default='warn' to suppress the warning message

df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/rdos_data_2020-2021.csv')

# Actual time that the prefix was seen was 1625668546
# Choose attack data of duration between given start time and end time is +2 hrs from the start time
condition = (df['start_posix_time'] >= from_time) & (df['end_posix_time'] <= until_time) 
df = df.loc[condition]

print("No. of target IP addresses for time %s and %s is %s" %(from_time, until_time, len(df)))

# The prefix you want to check against
# prefix = '1.165.12.0/24'

# prefix_seen_time = 1626285350


# Convert the prefix to an IPv4Network object
network = ipaddress.ip_network(prefix)

# Check if any IP falls within the prefix
df['ip_in_prefix'] = df['target_ip'].apply(lambda ip: ipaddress.ip_address(ip) in network)

# Filter the rows where the IP is in the prefix
matching_ips = df[df['ip_in_prefix'] == True]

# Print the matching IPs
if len(matching_ips) > 0:
    print("Found")
else: 
    print("Not found")

In [ ]:
#-------------------BLOCK #15--------------------------------------------
# -------------------------------------------------------------------------

# Same as above block where we check the announcement pattern of prefixes that were seen by NBIP (AS200020),
# in the file as200020_01_31_Dec_2020_rrc00.csv before and after certain hours
# Check BGP updates (announcements) for a few targeted IP addresses in RSDoS file
# Results are in office -> excel -> DDoS -> sheet -> NBIP 
# DIFFERENCE: Read input from a file and write results also to that file.

# ---------------ALGORITHM---------------
1. 

import pybgpstream
import datetime
import pytricia

prefix = "176.74.232.0/24" 
prefix_seen_time = 1625100237

# Please choose one of the three options at a time.
# 1. Check providers list before two hours the prefix is seen by a route collector 
# from_time = prefix_seen_time - 7201  
# until_time = prefix_seen_time - 1 

# 2. Check if a victim IP adddress exists for a prefix one day before within -1 hr and + 2 hr of BGP update time
from_time = prefix_seen_time - 86400 - 7200 
until_time = prefix_seen_time - 86400


# Convert to UTC time
from_time_utc = datetime.datetime.utcfromtimestamp(from_time)
until_time_utc = datetime.datetime.utcfromtimestamp(until_time)

# Convert UTC datetime to string
from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

print("Time %s and %s" %(from_time_utc_str, until_time_utc_str))


stream = pybgpstream.BGPStream(
    from_time=from_time_utc_str, until_time=until_time_utc_str,
    record_type="updates", # By default it is Update announcement 
    filter="prefix less "+prefix,
    project="ris"

  )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter("elemtype", "announcements")

pyt_v4 = pytricia.PyTricia() # For storing ipv4 prefixes

p = [] # List containing immediate provider
o = [] # Origin ASN

print("Extracting records..")

# Create a list of hash of prefix and provider
#[{"80.255.245.0/24": [200020]}, {"80.255.0.0/16": [49685, 174]}]

# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
        pfx = elem.fields["prefix"]
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]

        # Convert as_path into list
        as_path_list = as_path.split()
        orig = as_path_list[-1]
        
        # Store prefix and upstreams in a dictionary
        pfx_details = {}
        
        # Discard different forms of origins example AS-Set, confederation set/sequence. 
        # Take only a single AS origin which is common for a scrubbing activity
        if pfx != '0.0.0.0/0' and len(as_path_list) > 1:
            
            
#             Call a function to find provider because provider might not always be the second last AS in case of
#             AS path prepending and sibling ASes
              second_as = find_immediate_provider(as_path_list)
              
             # Find second ASN in an AS path
#             second_as = as_path_list[-2]
            
    
            # Extract its upstream provider to check if that one contains an scrubbers' ASN
            p.append(second_as)
            o.append(orig)
            
            
             # If the prefix is already in the tree, append the new provider
            if pyt_v4.has_key(pfx):
                pyt_v4[pfx]['provider'].append(second_as)
                
                # TODO: When I try to store origin here, it gives error
            else:
                # Add the prefix with a list of providers
                pyt_v4[pfx] = {'provider': [second_as]}
  
# Remove duplicates from provider lists (optional, if needed)
for prefix in pyt_v4:
    pyt_v4[prefix]['provider'] = list(set(pyt_v4[prefix]['provider']))

# Print prefixes and their providers
for prefix in pyt_v4:
    print (prefix,pyt_v4[prefix])
    

# Filter RSDoS data based on attack duration and check if a given prefix contains any IP address in that list
print("Finding matching IP address..")    

import ipaddress

pd.options.mode.chained_assignment = None  # default='warn' to suppress the warning message

df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/rdos_data_2020-2021.csv')

# Actual time that the prefix was seen was 1625668546
# Choose attack data of duration between given start time and end time is +2 hrs from the start time
condition = (df['start_posix_time'] >= from_time) & (df['end_posix_time'] <= until_time) 
df = df.loc[condition]

print("No. of target IP addresses for time %s and %s is %s" %(from_time, until_time, len(df)))

# The prefix you want to check against
# prefix = '1.165.12.0/24'

# prefix_seen_time = 1626285350


# Convert the prefix to an IPv4Network object
network = ipaddress.ip_network(prefix)

# Check if any IP falls within the prefix
df['ip_in_prefix'] = df['target_ip'].apply(lambda ip: ipaddress.ip_address(ip) in network)

# Filter the rows where the IP is in the prefix
matching_ips = df[df['ip_in_prefix'] == True]

# Print the matching IPs
if len(matching_ips) > 0:
    print("Found")
else: 
    print("Not found")

In [ ]:
#-------------------BLOCK #16--------------------------------------------
# -------------------------------------------------------------------------

# Filter RSDoS data based on attack duration and check if a given prefix contains any IP address in that list
import pandas as pd
import ipaddress

pd.options.mode.chained_assignment = None  # default='warn' to suppress the warning message

df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/rdos_data_2020-2021.csv')

# Actual time that the prefix was seen was 1625668546
# Choose attack data of duration between given start time and end time is +2 hrs from the start time
condition = (df['start_posix_time'] >= 1626191749) & (df['end_posix_time'] <= 1626198949) 
df = df.loc[condition]

# The prefix you want to check against
prefix = '1.165.12.0/24'

prefix_seen_time = 1626285350


# Convert the prefix to an IPv4Network object
network = ipaddress.ip_network(prefix)

# Check if any IP falls within the prefix
df['ip_in_prefix'] = df['target_ip'].apply(lambda ip: ipaddress.ip_address(ip) in network)

# Filter the rows where the IP is in the prefix
matching_ips = df[df['ip_in_prefix'] == True]

# Print the matching IPs
if len(matching_ips) > 0:
    print("Found")
else: 
    print("Not found")

In [3]:
#-------------------BLOCK #17--------------------------------------------
# -------------------------------------------------------------------------

# Detects if a given string is an IPv4 or IPv6
import ipaddress

def detect_ip_version(ip_network_str):
    try:
        ip_network = ipaddress.ip_network(ip_network_str, strict=False)
        if isinstance(ip_network, ipaddress.IPv4Network):
            return "IPv4"
        elif isinstance(ip_network, ipaddress.IPv6Network):
            return "IPv6"
    except ValueError:
        return "Invalid IP address or network"

In [18]:
#-------------------BLOCK #18--------------------------------------------
# -------------------------------------------------------------------------

# Program to find upstream provider using condition: 
# a. If the same origin AS is prepending, upstream AS is the next one after prepending. 
# b. If the origin AS has siblings, remove siblings. 
# CAVEAT: This program does not check as_path = [200, 300, 400, 400, 8074, 8075, 8075] where there is repeatation of 
# new ASes other than siblings

# Function to call an API to get the list of siblings for a given ASN (last origin)
def api_get_siblings(asn):
    
    as_rank = AsRank()
    siblings = as_rank.get_all_siblings(asn)

    return siblings[1] # It contains list of total siblings count and list of ASNs. We are concerned only with the latter.

# Function to find the immediate provider ASN
def find_immediate_provider(as_path):
    if len(as_path) < 2:
        return None  # Not enough ASNs in path to determine provider

    last_origin_asn = as_path[-1]  # The last ASN is the origin ASN

    # Check for sequentially repeated ASNs
    repeated_asn = None
    for i in range(len(as_path) - 1, 0, -1):
        if as_path[i-1] == as_path[i]:
            repeated_asn = as_path[i]
        else:
            # If we find an ASN that is not the same as the repeated ASN,
            # and we have found a repeated ASN, return the one before the repeated ASN
            if repeated_asn is not None:
                return as_path[i-1]  # This is the upstream provider
            break

# Do this check to call API only in case an origin has mutliple siblings
            
    # If no repeated ASN found or it's the only ASN in the path
    if repeated_asn is None:
        # Retrieve siblings for the last ASN (last origin in the AS path)
        try:
            print("AS rank API for getting siblings")
            sibling_list = api_get_siblings(last_origin_asn)
        except Exception as e:
            print(f"Error retrieving siblings: {e}")
            return None  # Handle API failure appropriately

        # Traverse the AS path 
        for i in range(len(as_path) - 1, 0, -1):
            if str(as_path[i-1]) not in sibling_list:
                return as_path[i-1]  # The first non-sibling ASN is the upstream provider

    return as_path[1]  # If all checks fail, return the second ASN by default


# Test cases based on your examples
as_path1 = [400, 100,  300, 300]  # Case 1: No repetition, no siblings
as_path = ["200", "300", "400", "400", "7584", "8074", "8075"]  # Case 2: Sequentially repeated, should return 300


# # Finding immediate providers
print("Immediate provider for AS path:", find_immediate_provider(as_path1))  # Expected Output: 300


Immediate provider for AS path: 100


In [1]:
#-------------------BLOCK #19--------------------------------------------
# -------------------------------------------------------------------------

# Method to get AS rank of all ASes from CAIDA AS rank API
# This part of the code is used from https://github.com/bgpkit/pyasrank

# MIT License

# Copyright (c) 2021 Mingwei Zhang

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.
import json
import logging
from datetime import datetime

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ASRANK_ENDPOINT = "https://api.asrank.caida.org/v2/graphql"


def ts_to_date_str(ts):
    """
    Convert timestamp to a date. This is used for ASRank API which only takes
    date strings with no time as parameters.api
    """
    return datetime.utcfromtimestamp(int(ts)).strftime("%Y-%m-%d")


class AsRank:
    """
    Utilities for using ASRank services
    """

    def __init__(self, max_ts=""):
        self.data_ts = None

        # various caches to avoid duplicate queries
        self.cache = None
        self.cone_cache = None
        self.neighbors_cache = None
        self.siblings_cache = None
        self.organization_cache = None

        self.queries_sent = 0

        self.session = None
        self._initialize_session()

        self.init_cache(max_ts)

    def _initialize_session(self):
        self.session = requests.Session()
        retries = Retry(total=5,
                        backoff_factor=1,
                        status_forcelist=[500, 502, 503, 504])
        self.session.mount(ASRANK_ENDPOINT, HTTPAdapter(max_retries=retries))

    def _close_session(self):
        if self.session:
            self.session.close()

    def _send_request(self, query):
        """
        send requests to ASRank endpoint
        :param query:
        :return:
        """

        r = self.session.post(url=ASRANK_ENDPOINT, json={'query': query})
        r.raise_for_status()
        self.queries_sent += 1
        return r

    def init_cache(self, ts):
        """
        Initialize the ASRank cache for the timestamp ts
        :param ts:
        :return:
        """
        self.cache = {}
        self.cone_cache = {}
        self.neighbors_cache = {}
        self.siblings_cache = {}
        self.organization_cache = {}
        self.queries_sent = 0
        if isinstance(ts, int):
            ts = ts_to_date_str(ts)

        ####
        # Try to cache datasets available before the given ts
        ####
        graphql_query = """
            {
              datasets(dateStart:"2000-01-01", dateEnd:"%s", sort:"-date", first:1){
                edges {
                  node {
                    date
                  }
                }
              }
            }
        """ % ts
        r = self._send_request(graphql_query)

        edges = r.json()['data']['datasets']['edges']
        if edges:
            self.data_ts = edges[0]["node"]["date"]
            return

        # if code reaches here, we have not found any datasets before ts. we should now try to find one after ts.
        # this is the best effort results
        logging.warning("cannot find dataset before date %s, looking for the closest one after it now" % ts)

        graphql_query = """
            {
              datasets(dateStart:"%s", sort:"date", first:1){
                edges {
                  node {
                    date
                  }
                }
              }
            }
        """ % ts
        r = self._send_request(graphql_query)
        edges = r.json()['data']['datasets']['edges']
        if edges:
            self.data_ts = edges[0]["node"]["date"]
            logging.warning("found closest dataset date to be %s" % self.data_ts)
            return
        else:
            raise ValueError("no datasets from ASRank available to use for tagging")

    def _query_asrank_for_asns(self, asns, chunk_size=100):
        asns = [str(asn) for asn in asns]
        asns_needed = [asn for asn in asns if asn not in self.cache]
        if not asns_needed:
            return

        # https://stackoverflow.com/a/312464/768793
        def chunks(lst, n):
            """Yield successive n-sized chunks from lst."""
            for i in range(0, len(lst), n):
                yield lst[i:i + n]

        for asns in chunks(asns_needed, chunk_size):

            graphql_query = """
                {
                  asns(asns: %s, dateStart: "%s", dateEnd: "%s", first:%d, sort:"-date") {
                    edges {
                      node {
                        date
                        asn
                        asnName
                        rank
                        organization{
                          country{
                            iso
                            name
                          }
                          orgName
                          orgId
                        } asnDegree {
                          provider
                          peer
                          customer
                          total
                          transit
                          sibling
                        }
                      }
                    }
                  }
                }
            """ % (json.dumps(asns), self.data_ts, self.data_ts, len(asns))
            r = self._send_request(graphql_query)
            try:
                for node in r.json()['data']['asns']['edges']:
                    data = node['node']
                    if data['asn'] not in self.cache:
                        if "asnDegree" in data:
                            degree = data["asnDegree"]
                            degree["provider"] = degree["provider"] or 0
                            degree["customer"] = degree["customer"] or 0
                            degree["peer"] = degree["peer"] or 0
                            degree["sibling"] = degree["sibling"] or 0
                            data["asnDegree"] = degree
                        self.cache[data['asn']] = data
                for asn in asns:
                    if asn not in self.cache:
                        self.cache[asn] = None
            except KeyError as e:
                logging.error("Error in node: {}".format(r.json()))
                logging.error("Request: {}".format(graphql_query))
                raise e

    

    def get_all_siblings(self, asn, skip_asrank_call=False):
        """
        get all siblings for an ASN
        :param asn: AS number to query for all siblings
        :param skip_asrank_call: skip asrank call if already done
        :return: a tuple of (TOTAL_COUNT, ASNs)
        """
        # FIXME: pagination does not work here. Example ASN5313.
        asn = str(asn)
        if asn in self.siblings_cache:
            return self.siblings_cache[asn]

        if not skip_asrank_call:
            self._query_asrank_for_asns([asn])

        if asn not in self.cache or self.cache[asn] is None:
            return 0, []
        asrank_info = self.cache[asn]
        if "organization" not in asrank_info or asrank_info["organization"] is None:
            return 0, []

        org_id = self.cache[asn]["organization"]["orgId"]

        if org_id in self.organization_cache:
            data = self.organization_cache[org_id]
        else:
            graphql_query = """
            {
            organization(orgId:"%s"){
              orgId,
              orgName,
              members{
                numberAsns,
                numberAsnsSeen,
                asns{totalCount,edges{node{asn,asnName}}}
              }
            }}        
            """ % org_id
            r = self._send_request(graphql_query)
            data = r.json()["data"]["organization"]
            self.organization_cache[org_id] = data

        if data is None:
            return 0, []

        total_cnt = data["members"]["asns"]["totalCount"]
        siblings = set()
        for sibling_data in data["members"]["asns"]["edges"]:
            siblings.add(sibling_data["node"]["asn"])
        if asn in siblings:
            siblings.remove(asn)
            total_cnt -= 1

        # NOTE: this assert can be wrong when number of siblings needs pagination
        # assert len(siblings) == total_cnt - 1

        siblings = list(siblings)
        self.neighbors_cache[asn] = (total_cnt, siblings)
        return total_cnt, siblings

In [11]:
asr = AsRank('2024-10-29')
siblings = asr.get_all_siblings(32787)[1]
# Convert strings to integers using
# list comprehension
siblings_int = [int(a) for a in siblings]
siblings_int

[16702,
 31984,
 23455,
 35993,
 18717,
 20189,
 36183,
 393560,
 22207,
 393234,
 17334,
 36029,
 18680,
 12222,
 30675,
 22452,
 26008,
 33047,
 16625,
 17204,
 23454,
 35994]

In [13]:
#-------------------BLOCK #20--------------------------------------------
# -------------------------------------------------------------------------
# Program to find siblings of an ASN based on CAIDA AS2Org data
# Read line from 95721 from the file /h # format:aut|changed|aut_name|org_id|opaque_id|source
# Example file data
import gzip
import json

import gzip
import json

# Function to find siblings of a given ASN in a .jsonl.gz file, starting from a specific line
def find_siblings(asn_to_find):
    # Create a dictionary to group ASNs by organizationId
    org_id_map = {}
    
    file_path = '/home/shyam/jupy/ddos_scrubber/data/20241001.as-org2info.jsonl.gz'

    start_line = 95721
        
    # Open and read the .jsonl.gz file
    with gzip.open(file_path, 'rt') as f:  # 'rt' is for reading text mode
        for current_line, line in enumerate(f):
            if current_line < start_line:
                continue  # Skip lines until reaching the desired start_line
            
            record = json.loads(line)  # Parse each line as JSON
            
            asn = record['asn']
            organization_id = record['organizationId']
            
            if organization_id not in org_id_map:
                org_id_map[organization_id] = []
            org_id_map[organization_id].append(asn)
    
    # Find the organizationId of the given ASN
    org_id_of_asn = None
    for org_id, asns in org_id_map.items():
        if asn_to_find in asns:
            org_id_of_asn = org_id
            break

    # If ASN is not found, return an empty list
    if org_id_of_asn is None:
        return []

    # Return all ASNs that share the same organizationId, excluding the given ASN
    siblings = [asn for asn in org_id_map[org_id_of_asn] if asn != asn_to_find]
    return siblings

# Test the function with a .jsonl.gz file, starting from line 10
asn_to_find = "32787"
siblings = find_siblings(asn_to_find)
print(f"Siblings of ASN {asn_to_find}: {siblings}")

Siblings of ASN 32787: ['12222', '16625', '16702', '17204', '17334', '18680', '18717', '20189', '22207', '22452', '23454', '23455', '26008', '30675', '31984', '33047', '35993', '35994', '36029', '36183', '393234', '393560']


In [ ]:
#-------------------BLOCK #21--------------------------------------------
# -------------------------------------------------------------------------
# Detects if a given string is an IPv4 or IPv6
import ipaddress

def detect_ip_version(ip_network_str):
    # Define variables to store instance and prefix length
    ipversion = "IPv4"
    pfx_len = "24"
    ip_network = ipaddress.ip_network(ip_network_str, strict=False)
    if isinstance(ip_network, ipaddress.IPv6Network):
        ipversion = "IPv6"
    pfx_len = ip_network.prefixlen
    return (ipversion, pfx_len)

# version, lent =  detect_ip_version("2a02:26f0:4200::/48")
